# 2.1 · SARIMAX y ETS sobre caudal del Genil

**Tiempo estimado:** 1 h.

**Objetivos.**

1. Ajustar un **SARIMAX** al caudal mensual del Genil con lluvia como exógena.
2. Ajustar un **ETS (Holt-Winters)** y comparar.
3. Producir un *forecast* a 24 meses con bandas de confianza.
4. Diagnosticar residuos.

**Datos:** Pinos-Genil (ROEA 5020) mensual + lluvia ERA5 mensual (Open-Meteo).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

## 1 · Preparación de datos

Pasamos a frecuencia mensual: caudal con **media**, lluvia con **suma**. Acotamos al periodo con buena cobertura (1995-2020).

In [ ]:
caudal_d = ud.cargar_caudal_genil(source="CEDEX")
lluvia_d = ud.cargar_lluvia_genil_diaria(fecha_inicio="1995-01-01", fecha_fin="2020-12-31")

caudal_m = caudal_d.resample("MS").mean().loc["1995":"2020"].interpolate("linear", limit=2).dropna()
lluvia_m = lluvia_d.resample("MS").sum().loc["1995":"2020"]

df = pd.DataFrame({"caudal": caudal_m, "lluvia": lluvia_m}).dropna()
print(f"Observaciones mensuales: {len(df)}")
print(df.describe().round(2))

## 2 · Split temporal

Train hasta dic-2017, test 2018-2020 (36 meses). **Sin barajar.**

In [ ]:
split = "2018-01-01"
train = df.loc[:split].iloc[:-1]
test = df.loc[split:]
print(f"Train: {train.index.min().date()} → {train.index.max().date()}   ({len(train)} obs)")
print(f"Test : {test.index.min().date()} → {test.index.max().date()}   ({len(test)} obs)")

## 3 · Identificación de órdenes (ACF / PACF)

Antes de ajustar SARIMA hay que decidir (p, d, q)(P, D, Q, s).

In [ ]:
y = train["caudal"]

# Tests sobre la serie y su diferencia estacional
print(f"ADF original          p={adfuller(y)[1]:.4f}")
print(f"ADF diff estacional   p={adfuller(y.diff(12).dropna())[1]:.4f}")
print(f"KPSS original         p={kpss(y, nlags='auto')[1]:.4f}")
print(f"KPSS diff estacional  p={kpss(y.diff(12).dropna(), nlags='auto')[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
plot_acf(y.diff(12).dropna(), lags=36, ax=axes[0])
plot_pacf(y.diff(12).dropna(), lags=36, ax=axes[1], method="ywm")
axes[0].set_title("ACF (Δ_12 caudal)")
axes[1].set_title("PACF (Δ_12 caudal)")
plt.tight_layout()

Hipótesis razonable: $(1, 0, 1)(1, 1, 1)_{12}$. Lo verificamos comparando algunas alternativas por AIC.

## 4 · Pequeña búsqueda en rejilla

In [ ]:
from itertools import product

exog_train = train["lluvia"].shift(1).fillna(method="bfill").values.reshape(-1, 1)

rejilla = list(product([0, 1, 2], [0, 1], [0, 1, 2], [0, 1], [0, 1], [0, 1]))
resultados = []
for p, d, q, P, D, Q in rejilla:
    try:
        m = SARIMAX(
            train["caudal"],
            exog=exog_train,
            order=(p, d, q),
            seasonal_order=(P, D, Q, 12),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        r = m.fit(disp=False, maxiter=50)
        resultados.append({"p": p, "d": d, "q": q, "P": P, "D": D, "Q": Q, "aic": r.aic})
    except Exception:
        continue

top = pd.DataFrame(resultados).sort_values("aic").head(5)
print("Top 5 por AIC:")
print(top.to_string(index=False))

## 5 · Ajuste final y diagnóstico

Cogemos el mejor por AIC y diagnosticamos residuos.

In [ ]:
best = top.iloc[0]
order = (int(best["p"]), int(best["d"]), int(best["q"]))
seas = (int(best["P"]), int(best["D"]), int(best["Q"]), 12)

modelo = SARIMAX(
    train["caudal"],
    exog=exog_train,
    order=order,
    seasonal_order=seas,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False, maxiter=200)
print(f"Modelo elegido: SARIMAX{order}{seas}")
print(f"AIC = {modelo.aic:.1f}   BIC = {modelo.bic:.1f}")

# Diagnóstico
fig = modelo.plot_diagnostics(figsize=(10, 5))
plt.tight_layout()

# Ljung-Box
lb = acorr_ljungbox(modelo.resid, lags=[12, 24], return_df=True)
print("\nLjung-Box (H0: residuos = ruido blanco):")
print(lb)

## 6 · Forecast 36 meses con bandas

Para el forecast necesitamos lluvia futura. En el mundo real eso vendría de un modelo meteorológico estacional o un escenario climático. Aquí usamos el **valor medio mensual histórico** como escenario neutro.

In [ ]:
# Lluvia futura: media climatológica del mes
clim_lluvia = train["lluvia"].groupby(train.index.month).mean()
exog_test = test.index.month.map(clim_lluvia).values.reshape(-1, 1)

fc = modelo.get_forecast(steps=len(test), exog=exog_test)
media = fc.predicted_mean
ci = fc.conf_int(alpha=0.20)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    train.index[-36:], train["caudal"].iloc[-36:], color="#1f6f8b", lw=1, label="train (último año)"
)
ax.plot(test.index, test["caudal"], color="black", lw=1.4, label="test (observado)")
ax.plot(test.index, media, color="#c2410c", lw=1.4, label="SARIMAX pronóstico")
ax.fill_between(
    test.index, ci.iloc[:, 0], ci.iloc[:, 1], color="#c2410c", alpha=0.2, label="IC 80%"
)
ax.set_ylabel("Q (m³/s)")
ax.legend()
ax.set_title(f"SARIMAX{order}{seas} — forecast test")
plt.tight_layout()

## 7 · ETS como comparación rápida

Holt-Winters aditivo: sin exógenas, sólo nivel + tendencia + estacionalidad.

In [ ]:
ets = ExponentialSmoothing(
    train["caudal"],
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit()
pred_ets = ets.forecast(len(test))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index[-36:], train["caudal"].iloc[-36:], color="#1f6f8b", lw=1, label="train")
ax.plot(test.index, test["caudal"], color="black", lw=1.4, label="test")
ax.plot(test.index, media, color="#c2410c", lw=1.4, ls="--", label="SARIMAX")
ax.plot(test.index, pred_ets, color="#16a34a", lw=1.4, ls="--", label="ETS Holt-Winters")
ax.set_ylabel("Q (m³/s)")
ax.legend()
plt.tight_layout()

## 8 · Métricas RMSE / MAE

Las métricas hidrológicas (NSE, KGE) se reservan para el notebook 04 — aquí baseline genérico.

In [ ]:
def rmse(y, yhat):
    return float(np.sqrt(((y - yhat) ** 2).mean()))


def mae(y, yhat):
    return float((y - yhat).abs().mean())


filas = []
for nombre, pred in [("SARIMAX", media), ("ETS", pred_ets)]:
    filas.append(
        {
            "modelo": nombre,
            "RMSE (m³/s)": rmse(test["caudal"], pred),
            "MAE (m³/s)": mae(test["caudal"], pred),
        }
    )
pd.DataFrame(filas).round(3)

## 9 · Ejercicios

1. **Forecast con lluvia perfecta.** Repite el pronóstico SARIMAX usando la **lluvia real** del test (no climatológica). ¿Cuánto mejora el RMSE?
2. **Comparativa de horizontes.** Calcula RMSE en h=1, h=3, h=6, h=12 meses. ¿Cómo crece el error con el horizonte?
3. **Sin exógena.** Ajusta el mismo SARIMA pero sin lluvia. Compara AIC y errores test.
4. **Modelo log.** Reajusta SARIMAX a `log(caudal+0.1)` y compara residuos. ¿Quedan más cerca de la normalidad?
5. **Reto.** Implementa una rejilla SARIMA con 2 covariables (lluvia mes anterior y lluvia mes actual). ¿Selecciona AIC un orden distinto?